# Spark Architecture — Deep Dive
## From Hadoop MapReduce → RDD → PySpark → What Happens Under the Hood

---

This notebook answers:
1. **Why did we move from Hadoop to Spark?** — What was broken
2. **What is RDD?** — Spark's core data structure
3. **What is PySpark?** — How Python talks to Spark
4. **Full Spark Architecture** — Driver, Cluster Manager, Executors, Jobs, Stages, Tasks
5. **1 Million Records walkthrough** — Exactly how Spark splits and parallelizes your data
6. **Live code** — Watch architecture in action with real output

> **Kernel:** Use `python311-pyspark` kernel for this notebook.

---
## PART 1 — The Problem With Hadoop MapReduce

### What was Hadoop?

Hadoop was the first big data framework (2006). It had two main parts:
- **HDFS** (Hadoop Distributed File System) — stores massive files split across many machines
- **MapReduce** — the computation engine to process those files

### How MapReduce worked

Every computation had to follow a rigid 2-step pattern:

```
Input Data (on disk)
    ↓
MAP phase     → each record gets transformed independently
    ↓
Shuffle/Sort  → data moved across network by key
    ↓
REDUCE phase  → grouped records get aggregated
    ↓
Output (written back to disk)
```

### The Critical Problem — Disk I/O at Every Step

```
MapReduce Job 1:
  Read from HDFS disk → Map → Reduce → Write to HDFS disk

MapReduce Job 2:
  Read from HDFS disk → Map → Reduce → Write to HDFS disk

MapReduce Job 3:
  Read from HDFS disk → Map → Reduce → Write to HDFS disk
```

**Every intermediate result hit the disk.** For a machine learning algorithm that needs 100 iterations, that's 200 disk reads + 200 disk writes.

### Real Numbers (Why This Mattered)

| Operation       | Speed         |
|-----------------|---------------|
| RAM access      | ~100 ns       |
| SSD read        | ~100,000 ns   |
| HDD read        | ~10,000,000 ns |
| Network         | ~500,000 ns   |

RAM is **100,000× faster** than a hard disk. MapReduce was throwing away this advantage after every single step.

### Other Hadoop Problems

| Problem | Impact |
|---------|--------|
| Only Map + Reduce operations | Could not express iterative algorithms (ML, graph processing) naturally |
| No interactive queries | You had to submit a full job and wait minutes |
| No streaming support | Batch only — no real-time processing |
| Verbose Java code | 50+ lines of Java for a simple word count |
| Slow startup | JVM + task launch overhead per job |

---

### Enter Apache Spark (2009, Berkeley AMPLab)

**Core insight:** Keep intermediate data **in memory (RAM)** instead of writing to disk between steps.

```
Hadoop MapReduce:
  Disk → Map → Disk → Reduce → Disk    (3 disk I/Os)

Spark equivalent:
  Disk → [Map → filter → groupBy → agg] → Disk   (1 disk read, 1 disk write)
                ↑ everything in between stays in RAM ↑
```

**Spark is 10–100× faster than Hadoop MapReduce** for most workloads.  
For iterative workloads (ML training), it can be **1000× faster**.

---
## PART 2 — RDD: Spark's Core Data Structure

### What is an RDD?

**RDD = Resilient Distributed Dataset**

Every word matters:

| Word | Meaning |
|------|---------|
| **Resilient** | Fault-tolerant — if a partition is lost (machine crash), Spark can recompute it from the lineage (the recipe of transformations) |
| **Distributed** | Data is split into **partitions** spread across multiple machines/cores |
| **Dataset** | A collection of records — could be numbers, strings, tuples, objects |

### The Physical Picture

Imagine you have 12 million order records in a file. Spark splits this file into partitions:

```
orders.csv  (12 million rows)
─────────────────────────────────
    Partition 0   [rows 0      – 999,999]    → loaded into Executor 1, Core 1
    Partition 1   [rows 1,000,000 – 1,999,999] → loaded into Executor 1, Core 2
    Partition 2   [rows 2,000,000 – 2,999,999] → loaded into Executor 2, Core 1
    Partition 3   [rows 3,000,000 – 3,999,999] → loaded into Executor 2, Core 2
    ...
    Partition 11  [rows 11,000,000 – 11,999,999] → loaded into Executor 4, Core 2
```

**Each partition is independent.** Executors process their own partition in parallel — they don't wait for each other.

### RDD Key Properties

```
RDD
├── Partitions          → list of partition objects (how data is split)
├── Dependencies        → parent RDD(s) this was derived from
├── Compute function    → function to compute records in a partition
├── Preferred locations → which machines hold the data (data locality)
└── Partitioner         → how keys are distributed (for key-value RDDs)
```

### Two Types of RDD Operations

#### 1. Transformations — lazy, return a new RDD
These do NOT run immediately. Spark just records the plan.

```python
rdd2 = rdd1.filter(lambda x: x > 0)    # lazy
rdd3 = rdd2.map(lambda x: x * 2)       # lazy
rdd4 = rdd3.groupByKey()               # lazy
# Nothing has executed yet!
```

#### 2. Actions — trigger actual execution
When an action is called, Spark looks at the full transformation chain and executes it.

```python
result = rdd4.count()    # ACTION → Spark now executes everything above
rdd4.collect()           # ACTION → brings all data to driver
rdd4.take(10)            # ACTION → brings first 10 records to driver
rdd4.saveAsTextFile()    # ACTION → writes to disk
```

### RDD Lineage — The Fault Tolerance Secret

```
rdd_raw   (loaded from disk)
    ↓  filter(amount > 0)
rdd_valid
    ↓  map(compute_total)
rdd_totals
    ↓  groupByKey()
rdd_grouped
    ↓  collect()         ← ACTION
```

If `rdd_totals` on Executor 3 crashes, Spark does NOT re-read the entire file.  
It just re-runs `filter` + `map` on the **one lost partition** using the lineage graph.  
This is why RDDs are called "resilient."

In [ ]:
import os, sys

# Must be set BEFORE any pyspark import
os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

print("Environment set.")
print(f"Python: {sys.executable}")

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkArchitectureDeepDive") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version  : {spark.version}")
print(f"Python version : {spark.sparkContext.pythonVer}")
print(f"Master         : {spark.sparkContext.master}")
print(f"Default parallelism (cores): {spark.sparkContext.defaultParallelism}")

---
## PART 3 — PySpark: Python API on Top of Spark

### What is PySpark?

Apache Spark is written in **Scala** (runs on JVM).  
PySpark is the **Python API** that lets you write Spark programs in Python.

### How PySpark Communicates With Spark (The Architecture)

```
Your Python Code (Jupyter / .py file)
         │
         │  You call: df.filter(...).groupBy(...).agg(...)
         │
         ▼
  ┌─────────────────────────────────┐
  │     PySpark Python Process      │  ← This is the DRIVER (Python side)
  │                                 │
  │   Py4J Gateway (port 25333)     │  ← Bridge between Python and JVM
  └────────────┬────────────────────┘
               │  Java/Scala method calls over local socket
               ▼
  ┌─────────────────────────────────┐
  │     JVM Process (SparkContext)  │  ← This is the DRIVER (Scala/JVM side)
  │                                 │
  │   - Builds execution plan       │
  │   - Talks to Cluster Manager    │
  │   - Tracks jobs/stages/tasks    │
  └────────────┬────────────────────┘
               │
               ▼  (sends tasks)
  ┌────────────────────────────────────────────────────────┐
  │                   CLUSTER MANAGER                      │
  │  (YARN / Standalone / Kubernetes / local[*])           │
  │  Decides: which machines run which tasks               │
  └──────┬──────────────┬──────────────┬───────────────────┘
         ▼              ▼              ▼
    Executor 1     Executor 2     Executor 3
    (Worker Node)  (Worker Node)  (Worker Node)
    [Core1][Core2] [Core1][Core2] [Core1][Core2]
```

### Py4J — The Bridge

**Py4J** is a library that allows Python programs to dynamically access Java objects in a JVM.

When you write:
```python
df.filter(col("amount") > 1000).count()
```

Python → Py4J → calls `JavaObject.filter(Column)` in the JVM → Spark runs in Scala.

**Important:** The actual data processing NEVER happens in Python.  
Python is only the **control plane** — it tells Spark what to do.  
The **data plane** (reading data, computing, shuffling) is 100% in the JVM executors.

### DataFrames Are Built On Top of RDDs

```
Your Python code writes:
    df.filter(col("status") == "delivered")

Spark internally:
    1. Parses this into a Logical Plan (abstract syntax tree)
    2. Catalyst Optimizer rewrites it into an optimized plan
    3. Generates bytecode (via Tungsten)
    4. Executes on RDD partitions in executors
```

You almost never write RDD code directly anymore — DataFrames give you SQL-like syntax and Spark handles the RDD layer for you.

---
## PART 4 — Full Spark Architecture: Every Component Explained

### The 5 Key Components

```
┌─────────────────────────────────────────────────────────────────────────┐
│                          SPARK APPLICATION                              │
│                                                                         │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │                         DRIVER PROGRAM                           │  │
│  │                                                                  │  │
│  │  SparkContext / SparkSession                                     │  │
│  │  ├── DAG Scheduler    → splits transformations into stages       │  │
│  │  ├── Task Scheduler   → sends tasks to executors                 │  │
│  │  ├── Block Manager    → tracks where data partitions live        │  │
│  │  └── Broadcast Mgr    → sends small lookup tables to all nodes   │  │
│  └──────────────────────────────┬───────────────────────────────────┘  │
│                                 │ negotiates resources                   │
│                    ┌────────────▼────────────┐                         │
│                    │    CLUSTER MANAGER      │                         │
│                    │                         │                         │
│                    │  • YARN (Hadoop)         │                         │
│                    │  • Kubernetes            │                         │
│                    │  • Spark Standalone      │                         │
│                    │  • local[*] (your laptop)│                         │
│                    └────────────┬────────────┘                         │
│                                 │ allocates                             │
│           ┌─────────────────────┼─────────────────────┐               │
│           ▼                     ▼                     ▼               │
│    ┌─────────────┐       ┌─────────────┐       ┌─────────────┐        │
│    │ EXECUTOR 1  │       │ EXECUTOR 2  │       │ EXECUTOR 3  │        │
│    │             │       │             │       │             │        │
│    │ [Task][Task]│       │ [Task][Task]│       │ [Task][Task]│        │
│    │             │       │             │       │             │        │
│    │ Cache/Memory│       │ Cache/Memory│       │ Cache/Memory│        │
│    │ Block Store │       │ Block Store │       │ Block Store │        │
│    └─────────────┘       └─────────────┘       └─────────────┘        │
└─────────────────────────────────────────────────────────────────────────┘
```

---

### Component 1: Driver

**What it is:** The process that runs YOUR code. When you run a Jupyter cell, the Driver is what's executing.

**What it does:**
- Creates the `SparkSession` / `SparkContext`
- Converts your DataFrame operations into a **DAG** (Directed Acyclic Graph) of transformations
- Splits that DAG into **Stages** and **Tasks**
- Sends tasks to executors
- Collects results when you call `.collect()` or `.show()`
- Tracks task status, retries failed tasks

**Critical rule:** The Driver is a single process. If it dies, the application dies.  
That's why you should NEVER do `df.collect()` on millions of rows — it sends all data to Driver memory and can crash it.

---

### Component 2: Cluster Manager

**What it is:** The resource allocator. It manages which machines are available and how much memory/CPU each executor gets.

| Cluster Manager | Used When |
|----------------|-----------|
| `local[*]`     | Development on your laptop — one JVM, no actual cluster |
| `local[4]`     | Simulate 4 cores locally |
| Spark Standalone | Small dedicated Spark cluster |
| YARN           | Spark on Hadoop cluster (production at most companies) |
| Kubernetes     | Spark on containers (modern cloud deployments) |
| Mesos          | Legacy (mostly replaced by Kubernetes) |

In our notebooks: `local[*]` means "use all cores on this machine as if they were separate executors."

---

### Component 3: Executor

**What it is:** The worker process. Each executor runs on one machine and has N cores (= N parallel tasks).

**What it does:**
- Receives tasks from the Driver
- Processes its assigned data partition
- Stores intermediate results in memory (or spills to disk if out of memory)
- Reports results back to Driver
- Stays alive for the entire duration of the application (unlike MapReduce which killed workers after each job)

**Key executor settings:**
```python
# When submitting a Spark job to a cluster:
spark-submit \
  --num-executors 10 \        # how many executor JVMs
  --executor-cores 4 \        # cores per executor (= parallel tasks per machine)
  --executor-memory 16g \     # RAM per executor
  my_job.py
```

---

### Component 4: Jobs, Stages, Tasks — The 3-Level Hierarchy

This is where Spark's actual execution happens. Understanding this is critical for debugging slow queries.

```
ONE Spark Application
└── Multiple JOBS (one per Action call)
    └── Each JOB = Multiple STAGES (split at shuffle boundaries)
        └── Each STAGE = Multiple TASKS (one per partition)
```

#### Job
Created every time you call an ACTION (`.count()`, `.show()`, `.collect()`, `.write()`).

```python
df.filter(...).groupBy(...).agg(...).show()   ← ONE job
df.count()                                     ← ANOTHER job
```

#### Stage
A stage is a group of transformations that can run without shuffling data across the network.  
**Stage boundaries happen at "wide transformations"** — operations that need data from multiple partitions.

```
Narrow transformations (no shuffle) → same stage:
  filter, map, withColumn, select, union

Wide transformations (shuffle required) → new stage boundary:
  groupBy, join, distinct, repartition, orderBy
```

#### Task
One task processes **one partition** on **one executor core**.  
If you have 200 partitions → Spark creates 200 tasks → runs them in parallel across available cores.

```
Stage 1 (filter + map):   200 tasks × 1 partition each  → runs on 8 cores = 25 rounds
Stage 2 (groupBy agg):    200 tasks × 1 partition each  → runs on 8 cores = 25 rounds
```

---
## PART 5 — 1 Million Records Walkthrough (Step by Step)

### The Scenario

You have **1,000,000 order records** in a CSV file.  
You want to: filter orders > ₹500, group by category, sum the revenue.

```python
result = df_orders \
    .filter(col("amount") > 500) \
    .groupBy("category") \
    .agg(sum("amount").alias("total_revenue"))
```

Your machine has **4 CPU cores**.

---

### Step 1 — SparkSession Created

```
Driver starts.
SparkContext connects to Cluster Manager (local[*] = 4 cores on your machine).
Cluster Manager allocates 1 Executor with 4 cores.
```

---

### Step 2 — Reading the File (Partitioning)

```
orders.csv  (1,000,000 rows, ~128MB)

Spark default partition size = 128MB
→ Spark creates 1 partition (fits in one block)

But local[*] with 4 cores → Spark creates at least 4 partitions to use all cores:

Partition 0: rows 1       – 250,000    → Core 1
Partition 1: rows 250,001 – 500,000    → Core 2
Partition 2: rows 500,001 – 750,000    → Core 3
Partition 3: rows 750,001 – 1,000,000  → Core 4

All 4 partitions load in parallel. Time ≈ time_to_read_1_partition.
```

---

### Step 3 — Stage 1: filter (Narrow Transformation)

```
filter(col("amount") > 500)

This is NARROW — each partition is processed independently, no data movement.

Core 1 processes Partition 0: scans 250,000 rows, keeps those with amount > 500
Core 2 processes Partition 1: scans 250,000 rows, keeps those with amount > 500
Core 3 processes Partition 2: scans 250,000 rows, keeps those with amount > 500
Core 4 processes Partition 3: scans 250,000 rows, keeps those with amount > 500

All 4 run simultaneously! Time ≈ time_to_filter_250,000_rows.
```

After filter (say 60% pass): ~600,000 rows remain, still in 4 partitions.

---

### Step 4 — Stage Boundary: groupBy triggers SHUFFLE

```
groupBy("category") is a WIDE transformation.
Rows with the same "category" might be in DIFFERENT partitions right now.

Before groupBy can aggregate, Spark must:
  1. Each core writes its filtered output to shuffle files (on disk)
  2. Rows get hashed by category:
       hash("Electronics") % 4 = partition 2
       hash("Clothing")    % 4 = partition 0
       hash("Books")       % 4 = partition 3
       hash("Groceries")   % 4 = partition 1
  3. Each new partition pulls rows from ALL old partitions over the network

This is called the SHUFFLE — the most expensive operation in Spark.

After shuffle:
  New Partition 0: ALL "Clothing"     rows from all 4 original partitions
  New Partition 1: ALL "Groceries"    rows from all 4 original partitions
  New Partition 2: ALL "Electronics"  rows from all 4 original partitions
  New Partition 3: ALL "Books"        rows from all 4 original partitions
```

---

### Step 5 — Stage 2: agg (Narrow after shuffle)

```
sum("amount") per category

Now each partition has ALL rows for one category.
No more data movement needed.

Core 1 sums all Clothing rows   → one number
Core 2 sums all Groceries rows  → one number
Core 3 sums all Electronics rows → one number
Core 4 sums all Books rows      → one number

4 rows result total.
```

---

### Step 6 — Action: result is returned to Driver

```
result.show() or result.collect()

4 summary rows come back to the Driver process.
Driver prints them or returns them to your Python code.
```

---

### Full Picture — Timeline

```
t=0s    Read CSV → 4 partitions created, loaded to memory
t=1s    Stage 1: filter runs on all 4 cores in parallel  ← PARALLEL
t=2s    Shuffle: each core writes shuffle files
t=3s    Shuffle: network transfer, new partitions assembled
t=4s    Stage 2: sum runs on all 4 cores in parallel     ← PARALLEL
t=5s    4 result rows returned to Driver

Total: 5 seconds

Compare Hadoop:
  1. Read 1M rows from disk
  2. Map (filter): write 600k rows to disk
  3. Read 600k rows from disk again
  4. Reduce (groupBy+sum): write 4 rows to disk
  Total: ~30-60 seconds (5-10x slower just for this query)
```

---

### What "local[*]" Means vs Real Cluster

```
local[*] (your laptop):
  - 1 JVM process = Driver + Executor combined
  - N cores = N parallel tasks
  - No network shuffle (memory copy instead)
  - No fault tolerance (no second machine to recover from)

Real cluster (YARN, 10 machines × 8 cores = 80 cores):
  - Separate Driver machine
  - 10 Executors, each on a different machine
  - Real network shuffle between machines
  - If 1 machine dies, Driver reassigns its tasks to other machines
  - Can process 20x more data in same time (80 cores vs 4 cores)
```

---
## PART 6 — Live Code: See Architecture in Action

Now we generate 1,000,000 records and actually watch Spark partition and process them.

In [ ]:
# ─── Generate 1,000,000 order records ─────────────────────────────────────────
import random
from pyspark.sql.functions import col, sum as spark_sum, count, avg, round as spark_round
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DateType

random.seed(42)

CATEGORIES = ["Electronics", "Clothing", "Books", "Groceries", "Furniture", "Sports"]
STATUSES   = ["delivered", "confirmed", "pending", "cancelled"]

N = 1_000_000

data = [
    (
        i,
        f"CUST-{random.randint(1, 100_000):06d}",
        random.choice(CATEGORIES),
        round(random.uniform(50, 10_000), 2),
        random.choice(STATUSES),
    )
    for i in range(1, N + 1)
]

schema = StructType([
    StructField("order_id",   IntegerType(), True),
    StructField("customer_id",StringType(),  True),
    StructField("category",   StringType(),  True),
    StructField("amount",     FloatType(),   True),
    StructField("status",     StringType(),  True),
])

df = spark.createDataFrame(data, schema=schema)

print(f"Total rows     : {N:,}")
print(f"Partitions     : {df.rdd.getNumPartitions()}")
print(f"Schema         :")
df.printSchema()

In [ ]:
# ─── Inspect how data is physically distributed across partitions ──────────────
# mapPartitionsWithIndex lets us see which partition each record lives in

def count_in_partition(partition_index, rows):
    row_count = sum(1 for _ in rows)
    yield (partition_index, row_count)

partition_counts = df.rdd.mapPartitionsWithIndex(count_in_partition).collect()

print("Partition Distribution:")
print(f"{'Partition':<12} {'Row Count':>12} {'Visual'}")
print("-" * 55)
for pid, cnt in sorted(partition_counts):
    bar = "█" * int(cnt / 20000)
    print(f"Partition {pid:<3} {cnt:>12,}  {bar}")

total_from_partitions = sum(c for _, c in partition_counts)
print("-" * 55)
print(f"{'TOTAL':<12} {total_from_partitions:>12,}")
print(f"\nSpark spread {total_from_partitions:,} rows across {len(partition_counts)} partitions.")

In [ ]:
# ─── Stage 1: Narrow transformation (filter) — no shuffle ─────────────────────
# Each partition independently filters rows. All partitions run in parallel.
import time

print("=" * 60)
print("STAGE 1: filter(amount > 500)  ← NARROW transformation")
print("         Each partition works independently, no data movement")
print("=" * 60)

t0 = time.time()
df_filtered = df.filter(col("amount") > 500)

# count() is an ACTION — triggers Stage 1 to run
count_before = N
count_after  = df_filtered.count()
t1 = time.time()

print(f"\nBefore filter : {count_before:>10,} rows")
print(f"After  filter : {count_after:>10,} rows")
print(f"Rows removed  : {count_before - count_after:>10,}")
print(f"Pass rate     : {count_after/count_before*100:.1f}%")
print(f"Time (count)  : {t1-t0:.2f}s")
print(f"\nPartitions after filter : {df_filtered.rdd.getNumPartitions()}")
print("(Same number — filter is narrow, partitions don't change)")

In [ ]:
# ─── Stage Boundary: groupBy triggers SHUFFLE ─────────────────────────────────
# Rows with the same category must be co-located in the same partition.
# Spark writes shuffle files and reassigns rows by hash(category).

print("=" * 60)
print("STAGE BOUNDARY: groupBy('category')  ← WIDE transformation")
print("                Triggers shuffle — data moves across partitions")
print("=" * 60)

t0 = time.time()
df_grouped = df_filtered.groupBy("category").agg(
    count("*").alias("order_count"),
    spark_round(spark_sum("amount"), 2).alias("total_revenue"),
    spark_round(avg("amount"), 2).alias("avg_order_value"),
)

# show() is the ACTION that triggers the full pipeline:
# Stage 1: filter  (narrow)
# Shuffle: groupBy (wide)
# Stage 2: agg     (narrow)
df_grouped.orderBy("category").show(truncate=False)
t1 = time.time()

print(f"\nTime for full pipeline (filter + shuffle + agg): {t1-t0:.2f}s")
print(f"\nPartitions after groupBy : {df_grouped.rdd.getNumPartitions()}")
print("(Controlled by spark.sql.shuffle.partitions = 4 in our config)")

In [ ]:
# ─── Inspect the Execution Plan — see exactly what Spark is doing ──────────────
# explain() shows the physical plan: what operations, in what order, on what data

print("=" * 60)
print("EXECUTION PLAN (explain())")
print("This is what Spark actually does internally.")
print("Read bottom to top — that's the execution order.")
print("=" * 60)

df_filtered.groupBy("category").agg(
    count("*").alias("order_count"),
    spark_round(spark_sum("amount"), 2).alias("total_revenue"),
).explain(mode="simple")

### How to Read the Execution Plan

```
== Physical Plan ==
AdaptiveSparkPlan (top — final output)
└── HashAggregate (final aggregation after shuffle)
    └── Exchange hashpartitioning(category, 4)   ← THIS IS THE SHUFFLE
        └── HashAggregate (partial agg per partition before shuffle)
            └── Filter (amount > 500.0)          ← our filter
                └── Scan (read the data)         ← bottom — starts here
```

**Key insight — Partial aggregation (an optimization):**  
Before the shuffle, Spark does a `partial sum` per category within each partition.  
This reduces the amount of data that needs to move across the network.

```
Before optimization:
  Partition 0 has 250,000 Electronics rows → all 250,000 move during shuffle

After optimization (partial agg):
  Partition 0 sums Electronics first → 1 number moves during shuffle
  Much less network traffic!
```

This optimization is called **map-side aggregation** or **partial aggregation** and Spark does it automatically.

In [ ]:
# ─── Demonstrate Lazy Evaluation ──────────────────────────────────────────────
# These transformations build a plan but do NOT execute — no data is read yet.
import time

print("Building transformation chain (lazy — no execution yet)...")
t0 = time.time()

# Chain of transformations — ALL LAZY
lazy_chain = (
    df
    .filter(col("status").isin("delivered", "confirmed"))
    .filter(col("amount") > 1000)
    .withColumn("revenue_tier",
        col("amount").cast("int") // 1000 * 1000
    )
    .groupBy("category", "revenue_tier")
    .agg(
        count("*").alias("orders"),
        spark_round(spark_sum("amount"), 2).alias("revenue"),
    )
    .orderBy("category", "revenue_tier")
)

t1 = time.time()
print(f"Time to build plan : {t1-t0:.4f}s  ← almost zero, nothing ran yet")
print(f"Type               : {type(lazy_chain)}")
print()

print("Now calling .show() — THIS triggers actual execution:")
t2 = time.time()
lazy_chain.show(20, truncate=False)
t3 = time.time()
print(f"Time to EXECUTE : {t3-t2:.2f}s  ← all the work happens here")

In [ ]:
# ─── Demonstrate RDD directly — the layer underneath DataFrames ───────────────
# DataFrames are built on top of RDDs.
# You can always drop to RDD level to see what's happening.

print("=" * 60)
print("RDD LAYER — underneath every DataFrame")
print("=" * 60)

# Get the underlying RDD from a DataFrame
rdd = df.rdd

print(f"\nDataFrame has {df.rdd.getNumPartitions()} partitions")
print(f"RDD type: {type(rdd)}")

# mapPartitionsWithIndex — run a function on each partition, get partition ID
def partition_summary(partition_index, rows):
    rows_list = list(rows)
    total     = len(rows_list)
    amounts   = [r.amount for r in rows_list]
    min_amt   = min(amounts) if amounts else 0
    max_amt   = max(amounts) if amounts else 0
    categories = set(r.category for r in rows_list)
    yield {
        "partition":  partition_index,
        "row_count":  total,
        "min_amount": round(min_amt, 2),
        "max_amount": round(max_amt, 2),
        "categories": sorted(categories),
    }

summaries = rdd.mapPartitionsWithIndex(partition_summary).collect()

print("\nPer-partition summary (RDD.mapPartitionsWithIndex):")
print("-" * 60)
for s in summaries:
    print(f"  Partition {s['partition']}: {s['row_count']:>8,} rows | "
          f"amount [{s['min_amount']:>7.1f} – {s['max_amount']:>8.1f}] | "
          f"categories: {s['categories']}")

print("\nNote: Each partition has ALL 6 categories because data is range-partitioned")
print("(created from Python list → rows assigned round-robin to partitions)")

In [ ]:
# ─── See partition distribution AFTER shuffle (groupBy) ───────────────────────
# After groupBy, each partition should have specific categories only (hash-partitioned)

print("=" * 60)
print("AFTER SHUFFLE: partition distribution by category")
print("groupBy hashes category → each partition gets specific categories")
print("=" * 60)

df_after_shuffle = df.groupBy("category").agg(
    count("*").alias("order_count"),
    spark_round(spark_sum("amount"), 2).alias("total_revenue"),
)

def shuffle_partition_info(partition_index, rows):
    rows_list  = list(rows)
    categories = [r.category for r in rows_list]
    yield (partition_index, sorted(set(categories)), len(rows_list))

shuffle_info = df_after_shuffle.rdd.mapPartitionsWithIndex(shuffle_partition_info).collect()

print(f"\nPartitions after groupBy: {df_after_shuffle.rdd.getNumPartitions()}")
print("(spark.sql.shuffle.partitions = 4)\n")
for pid, cats, cnt in sorted(shuffle_info):
    print(f"  Partition {pid}: {cnt} row(s) → categories: {cats}")

print()
print("KEY OBSERVATION: After shuffle, each partition has SPECIFIC categories.")
print("All rows for a given category are in EXACTLY ONE partition.")
print("This is what makes the aggregation correct — no missed rows.")

In [ ]:
# ─── Demonstrate caching: keep RDD in memory for reuse ────────────────────────
# If you use a DataFrame multiple times, cache it to avoid recomputing.

print("=" * 60)
print("CACHING — avoid recomputing the same data twice")
print("=" * 60)

# Without cache — filter + count runs TWICE
t0 = time.time()
expensive = df.filter(col("status") == "delivered").filter(col("amount") > 2000)
c1 = expensive.count()           # triggers full scan + filter
c2 = expensive.agg(spark_sum("amount")).collect()[0][0]  # triggers AGAIN
t1 = time.time()
print(f"\nWithout cache: 2 actions = 2 full scans")
print(f"  count = {c1:,}, sum = {c2:,.0f}")
print(f"  Time  = {t1-t0:.2f}s")

# With cache — filter runs ONCE, result stays in memory
t2 = time.time()
expensive_cached = df.filter(col("status") == "delivered").filter(col("amount") > 2000)
expensive_cached.cache()          # mark for caching
c3 = expensive_cached.count()    # first action: computes AND caches
c4 = expensive_cached.agg(spark_sum("amount")).collect()[0][0]  # reads from cache
t3 = time.time()
print(f"\nWith cache: 2 actions, but only 1 full scan")
print(f"  count = {c3:,}, sum = {c4:,.0f}")
print(f"  Time  = {t3-t2:.2f}s")

expensive_cached.unpersist()
print("\n(Cache freed with .unpersist())")
print("\nRule of thumb: cache a DataFrame if you use it in 2+ downstream operations.")

In [ ]:
# ─── Demonstrate repartition vs coalesce ──────────────────────────────────────
# repartition = FULL shuffle, can increase or decrease partition count
# coalesce    = no shuffle, can only decrease (merges partitions)

print("=" * 60)
print("REPARTITION vs COALESCE")
print("=" * 60)

print(f"\nOriginal partitions : {df.rdd.getNumPartitions()}")

# Repartition to 8 — triggers a full shuffle, redistributes rows evenly
df_8 = df.repartition(8)
print(f"After repartition(8): {df_8.rdd.getNumPartitions()} partitions  ← full shuffle, data balanced")

# Repartition by column — hash partitioning (great before joins)
df_by_cat = df.repartition(4, "category")
print(f"After repartition(4, 'category'): {df_by_cat.rdd.getNumPartitions()} partitions  ← all rows for same category in same partition")

# Coalesce — just merges existing partitions, no data movement
df_2 = df_8.coalesce(2)
print(f"After coalesce(2): {df_2.rdd.getNumPartitions()} partitions  ← no shuffle, just merged")

print("""
When to use each:
  repartition(N)          → before a large join or groupBy with many partitions
  repartition(N, col)     → before joining on a specific key (avoids shuffle at join time)
  coalesce(N)             → before writing output (fewer files = better)
  coalesce(1)             → write a single output file (careful: all data to 1 executor!)
""")

---
## PART 7 — Catalyst Optimizer: How Spark Makes Your Code Faster Automatically

When you write a DataFrame query, Spark doesn't execute it as-is.  
It passes your query through the **Catalyst Optimizer** — a rule-based + cost-based query optimizer.

```
Your Python Code
        │
        ▼
┌───────────────────────────────────────┐
│  Unresolved Logical Plan              │  ← parsed, column names not yet resolved
│  (Abstract Syntax Tree)               │
└──────────────┬────────────────────────┘
               │  Analysis (resolve column names/types)
               ▼
┌───────────────────────────────────────┐
│  Resolved Logical Plan                │
└──────────────┬────────────────────────┘
               │  Logical Optimization (Catalyst rules)
               ▼
┌───────────────────────────────────────┐
│  Optimized Logical Plan               │
│                                       │
│  Rules applied:                       │
│  • Predicate Pushdown                 │  ← move filters closer to data source
│  • Column Pruning                     │  ← drop columns not used downstream
│  • Constant Folding                   │  ← evaluate 2+3 at plan time, not runtime
│  • Join Reordering                    │  ← smallest table on right side of join
└──────────────┬────────────────────────┘
               │  Physical Planning
               ▼
┌───────────────────────────────────────┐
│  Physical Plan(s)                     │
│                                       │
│  Multiple physical strategies tried:  │
│  • BroadcastHashJoin vs SortMergeJoin │
│  • HashAgg vs SortAgg                 │
└──────────────┬────────────────────────┘
               │  Cost Model selects best
               ▼
┌───────────────────────────────────────┐
│  Selected Physical Plan               │
└──────────────┬────────────────────────┘
               │  Tungsten code generation
               ▼
        Bytecode → Execute on RDD partitions
```

### Example: Predicate Pushdown

```python
# What you write:
df.select("order_id", "amount") \
  .filter(col("amount") > 500)

# What Catalyst does:
# Pushes the filter DOWN to happen during the scan.
# Only rows with amount > 500 are even read into memory.
# Less data = faster execution.
```

### Example: Column Pruning

```python
# What you write:
df.select("order_id", "amount").groupBy("category").agg(sum("amount"))
# Wait — groupBy needs "category" but select didn't include it.
# Catalyst rewrites the select to include "category" automatically.
# More importantly: it DROPS all other columns (customer_id, status)
# so they're never loaded from storage.
```

In [ ]:
# ─── Show all 4 plan levels with explain("extended") ─────────────────────────

print("=" * 60)
print("FULL PLAN: Unresolved → Analyzed → Optimized → Physical")
print("=" * 60)

df.filter(col("amount") > 500) \
  .select("order_id", "category", "amount") \
  .groupBy("category") \
  .agg(spark_round(spark_sum("amount"), 2).alias("revenue")) \
  .explain("extended")

---
## PART 8 — Complete Architecture Summary

### Everything Together

```
┌──────────────────────────────────────────────────────────────────────────┐
│                     SPARK APPLICATION LIFECYCLE                          │
│                                                                          │
│  1. You write Python (PySpark) code                                      │
│     df.filter(...).groupBy(...).agg(...).show()                          │
│                        │                                                 │
│  2. Py4J sends it to JVM (SparkContext in Driver)                        │
│                        │                                                 │
│  3. Catalyst Optimizer builds + optimizes execution plan                 │
│     Unresolved Plan → Analyzed → Optimized → Physical Plan               │
│                        │                                                 │
│  4. DAG Scheduler splits plan into Stages                                │
│     Stage boundary = shuffle (groupBy, join, distinct, orderBy)          │
│                        │                                                 │
│  5. Task Scheduler assigns Tasks to Executors                            │
│     1 Task = 1 Partition = 1 Executor Core                               │
│                        │                                                 │
│  6. Executors process partitions in parallel                             │
│     Each executor: read data → transform → (shuffle if needed) → agg    │
│                        │                                                 │
│  7. Results flow back to Driver                                          │
│     .show() → 20 rows sent to Driver                                     │
│     .collect() → ALL rows sent to Driver (dangerous on large data!)      │
│     .write() → results written to storage (no data sent to Driver)       │
└──────────────────────────────────────────────────────────────────────────┘
```

### Narrow vs Wide — The Rule That Governs Performance

| Transformation | Type | Shuffle? | Stage boundary? |
|---------------|------|----------|-----------------|
| `filter`      | Narrow | No | No |
| `select`      | Narrow | No | No |
| `withColumn`  | Narrow | No | No |
| `map`         | Narrow | No | No |
| `union`       | Narrow | No | No |
| `groupBy`     | **Wide** | **Yes** | **Yes** |
| `join`        | **Wide** | **Yes** | **Yes** |
| `distinct`    | **Wide** | **Yes** | **Yes** |
| `orderBy`     | **Wide** | **Yes** | **Yes** |
| `repartition` | **Wide** | **Yes** | **Yes** |

### Key Numbers to Remember

| Config | Default | When to change |
|--------|---------|----------------|
| `spark.sql.shuffle.partitions` | 200 | Lower to 4-8 for small local data |
| `spark.default.parallelism` | 2× cores | Increase for more parallel tasks |
| Max partition size | 128MB | Increase for larger files |
| Executor memory | 1g | Increase if you see OOM errors |

### Anti-Patterns to Avoid

```python
# BAD — brings all 1M rows to Driver, likely crashes
df.collect()

# BAD — forces 1 partition, kills parallelism
df.coalesce(1).groupBy(...).agg(...)   # groupBy on 1 partition = single-threaded!

# BAD — calling count() multiple times on same DataFrame
n1 = df.filter(...).count()   # full scan
n2 = df.filter(...).count()   # ANOTHER full scan
# Fix: cache() the filtered df first

# GOOD
df_filtered = df.filter(...).cache()
n1 = df_filtered.count()
n2 = df_filtered.agg(sum("amount")).collect()
df_filtered.unpersist()
```

In [ ]:
# ─── Final summary: inspect this SparkSession's runtime config ────────────────

print("=" * 60)
print("THIS SESSION — Runtime Configuration")
print("=" * 60)

sc = spark.sparkContext

print(f"\nApplication ID     : {sc.applicationId}")
print(f"App name           : {sc.appName}")
print(f"Master             : {sc.master}")
print(f"Spark version      : {spark.version}")
print(f"Python version     : {sc.pythonVer}")
print(f"Default parallelism: {sc.defaultParallelism}  (= number of CPU cores available)")
print(f"Shuffle partitions : {spark.conf.get('spark.sql.shuffle.partitions')}")

print("\n" + "=" * 60)
print("Spark UI (while session is running):")
print("  http://localhost:4040")
print("  → Jobs tab:   see every action, its duration, stages")
print("  → Stages tab: see tasks, shuffle read/write, duration per stage")
print("  → Storage:    see cached DataFrames")
print("  → SQL tab:    see physical plans visually (DAG graph)")
print("=" * 60)

spark.stop()
print("\nSparkSession stopped.")